Environment & Cleanup

In [ ]:
import os
import shutil

# --- CONFIGURABLE PATHS ---
DATASET_PATH = r"C:/TTS-glowtts/XTTS_Dataset" 
OUTPUT_PATH = r"C:/TTS-glowtts/output"
BASE_MODEL_PATH = r"C:/TTS-Glow/base_model"
os.makedirs(BASE_MODEL_PATH, exist_ok=True)
LOCAL_CHECKPOINT = os.path.join(BASE_MODEL_PATH, "best_model.pth.tar")
LOCAL_CONFIG = os.path.join(BASE_MODEL_PATH, "config.json")


In [ ]:
from trainer import Trainer, TrainerArgs
from TTS.tts.datasets import load_tts_samples
from TTS.tts.configs.shared_configs import BaseDatasetConfig
from TTS.utils.audio import AudioProcessor

try:
    from TTS.tts.models.glow_tts import GlowTTS as GlowTts
    from TTS.tts.configs.glow_tts_config import GlowTtsConfig
except ImportError:
    try:
        # Some versions use this path
        from TTS.tts.models.glow_tts import GlowTts
        from TTS.tts.models.glow_tts import GlowTtsConfig
    except ImportError:
        # Final fallback for older versions
        from TTS.tts.models.glow_tts import GlowTTS as GlowTts
        from TTS.tts.models.glow_tts import GlowTTSConfig as GlowTtsConfig

print(" Imports successful!")

from TTS.tts.utils.text.tokenizer import TTSTokenizer

def run_glow_training():
    # 1. Initialize Configuration
    config = GlowTtsConfig(
        batch_size=16,
        lr=1e-5,
        eval_batch_size=8,
        num_loader_workers=0,
        num_eval_loader_workers=0,
        run_eval=True,
        test_delay_epochs=-1,
        epochs=100,
        text_cleaner="english_cleaners",
        use_phonemes=False,
        print_step=10,
        print_eval=True,
        output_path=OUTPUT_PATH,
    )

    # 2. Setup Dataset
    dataset_config = BaseDatasetConfig(
        formatter="ljspeech",
        meta_file_train="metadata_train.csv",
        path=DATASET_PATH
    )

    # Initialize the Tokenizer from the config
    # This tells the model how many characters (num_chars) to expect
    tokenizer, config = TTSTokenizer.init_from_config(config)

    train_samples, eval_samples = load_tts_samples(
        dataset_config,
        eval_split=True,
        eval_split_max_size=64,
        eval_split_size=0.1
    )

    # 3. Initialize Audio Processor & Model
    ap = AudioProcessor.init_from_config(config)
    
    # Pass the tokenizer to the model initialization
    model = GlowTts(config, ap, tokenizer=tokenizer, speaker_manager=None)
    

    if os.path.exists(LOCAL_CHECKPOINT):
        print(f" Loading Italian Base Model: {LOCAL_CHECKPOINT}")
        model.load_checkpoint(config, LOCAL_CHECKPOINT, eval=False, strict=False)
    else:
        print(" ERROR: Could not find the base model files. Check your paths!")
    
    # 4. Initialize Trainer
    trainer = Trainer(
        TrainerArgs(),
        config,
        output_path=OUTPUT_PATH,
        model=model,
        train_samples=train_samples,
        eval_samples=eval_samples,
    )

    print(" Glow-TTS training starting!")
    trainer.fit()

c:\TTS-glowtts\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Imports successful!


In [ ]:
import os
import json
import shutil

# --- CONFIGURABLE PATHS ---
DATASET_PATH = r"C:/TTS-glowtts/XTTS_Dataset"
OUTPUT_PATH = r"C:/TTS-glowtts/output"
BASE_MODEL_PATH = r"C:/TTS-glowtts/Base_Model"
LOCAL_CHECKPOINT = os.path.join(BASE_MODEL_PATH, "model_file.pth")
LOCAL_CONFIG = os.path.join(BASE_MODEL_PATH, "config.json")

# 1. Handle Imports with fallbacks
from trainer import Trainer, TrainerArgs
from TTS.tts.datasets import load_tts_samples
from TTS.tts.configs.shared_configs import BaseDatasetConfig
from TTS.utils.audio import AudioProcessor
from TTS.tts.utils.text.tokenizer import TTSTokenizer

try:
    from TTS.tts.configs.glow_tts_config import GlowTtsConfig
except ImportError:
    from TTS.tts.models.glow_tts import GlowTTSConfig as GlowTtsConfig

from TTS.tts.models.glow_tts import GlowTTS as GlowTts
import librosa
import os

dataset_path = r"C:/TTS-glowtts/XTTS_Dataset"
# Loop through files and try to get duration
for file in os.listdir(dataset_path):
    if file.endswith(".wav"):
        try:
            full_path = os.path.join(dataset_path, file)
            duration = librosa.get_duration(filename=full_path)
        except Exception as e:
            print(f" FAILED: {file} - Reason: {e}")
def run_training():
    # 2. Load Config from JSON file manually
    with open(LOCAL_CONFIG, 'r', encoding='utf-8') as f:
        config_dict = json.load(f)

    # 3. Initialize the proper Coqpit object and fill it
    config = GlowTtsConfig()
    config.from_dict(config_dict)

    # 4. Override for Fine-Tuning
    config.output_path = OUTPUT_PATH
    config.batch_size = 16 
    config.epochs = 100
    config.lr = 1e-5
    config.num_loader_workers = 0       
    config.num_eval_loader_workers = 0
    config.use_phonemes = True          

    # 5. Setup Dataset
    dataset_config = BaseDatasetConfig(
        formatter="ljspeech",
        meta_file_train="metadata_train.csv",
        path=DATASET_PATH
    )
    config.datasets = [dataset_config]

    # 6. Initialize Components
    ap = AudioProcessor.init_from_config(config)
    tokenizer, config = TTSTokenizer.init_from_config(config)
    
    model = GlowTts(config, ap, tokenizer=tokenizer)

    # 7. Load Checkpoint
    if os.path.exists(LOCAL_CHECKPOINT):
        print(f" Loading Weights from: {LOCAL_CHECKPOINT}")
        model.load_checkpoint(config, LOCAL_CHECKPOINT, eval=False)
    else:
        raise FileNotFoundError(f"Checkpoint not found at {LOCAL_CHECKPOINT}")

    # 8. Start Trainer
    trainer = Trainer(
        TrainerArgs(),
        config,
        output_path=OUTPUT_PATH,
        model=model,
        train_samples=load_tts_samples(dataset_config, eval_split=True)[0],
        eval_samples=load_tts_samples(dataset_config, eval_split=True)[1],
    )

    trainer.fit()

if __name__ == "__main__":
    run_training()

🧠 Loading Weights from: C:/TTS-glowtts/Base_Model\model_file.pth


 > Training Environment:
 | > Backend: Torch
 | > Mixed precision: True
 | > Precision: fp16
 | > Current device: 0
 | > Num. of GPUs: 1
 | > Num. of CPUs: 12
 | > Num. of Torch Threads: 6
 | > Torch seed: 54321
 | > Torch CUDNN: True
 | > Torch CUDNN deterministic: False
 | > Torch CUDNN benchmark: False
 | > Torch TF32 MatMul: False
 > Start Tensorboard: tensorboard --logdir=C:/TTS-glowtts/output\glow-tts-rel_pos_transformer-February-22-2026_06+45PM-0000000

 > Model has 28609873 parameters

 > EPOCH: 0/100
 --> C:/TTS-glowtts/output\glow-tts-rel_pos_transformer-February-22-2026_06+45PM-0000000

 > TRAINING (2026-02-22 18:45:38) 

   --> TIME: 2026-02-22 18:45:41 -- STEP: 0/233 -- GLOBAL_STEP: 0
     | > current_lr: 2.5e-09 
     | > step_time: 2.4248  (2.424833297729492)
     | > loader_time: 1.3187  (1.3187494277954102)

 [!] `train_step()` retuned `None` outputs. Skipping training step.
 [!] `train_step()` retuned `None` outputs. Skipping training step.
 [!] `train_step()` retuned


  --> EVAL PERFORMANCE
     | > avg_loader_time: 0.2883962392807007 (+0)
     | > avg_loss: 2.41900897026062 (+0)
     | > avg_log_mle: 0.7810281813144684 (+0)
     | > avg_loss_dur: 1.6379806995391846 (+0)

 > BEST MODEL : C:/TTS-glowtts/output\glow-tts-rel_pos_transformer-February-22-2026_06+45PM-0000000\best_model_233.pth

 > EPOCH: 1/100
 --> C:/TTS-glowtts/output\glow-tts-rel_pos_transformer-February-22-2026_06+45PM-0000000

 > TRAINING (2026-02-22 18:50:08) 

   --> TIME: 2026-02-22 18:50:15 -- STEP: 17/233 -- GLOBAL_STEP: 250
     | > loss: 2.355194568634033  (2.4623607326956356)
     | > log_mle: 0.5712673664093018  (0.6189554929733276)
     | > loss_dur: 1.7839272022247314  (1.8434052677715527)
     | > amp_scaler: 4096.0  (4096.0)
     | > grad_norm: tensor(15.4398, device='cuda:0')  (tensor(18.2379, device='cuda:0'))
     | > current_lr: 2.5e-09 
     | > step_time: 0.2934  (0.31189629610847025)
     | > loader_time: 0.1461  (0.11370276002322927)


   --> TIME: 2026-02-22 1


  --> EVAL PERFORMANCE
     | > avg_loader_time: 0.14700686931610107 (-0.1413893699645996)
     | > avg_loss: 2.4184670448303223 (-0.0005419254302978516)
     | > avg_log_mle: 0.7805075943470001 (-0.0005205869674682617)
     | > avg_loss_dur: 1.6379594802856445 (-2.1219253540039062e-05)

 > BEST MODEL : C:/TTS-glowtts/output\glow-tts-rel_pos_transformer-February-22-2026_06+45PM-0000000\best_model_466.pth

 > EPOCH: 2/100
 --> C:/TTS-glowtts/output\glow-tts-rel_pos_transformer-February-22-2026_06+45PM-0000000

 > TRAINING (2026-02-22 18:53:05) 

   --> TIME: 2026-02-22 18:53:10 -- STEP: 9/233 -- GLOBAL_STEP: 475
     | > loss: 2.438976526260376  (2.547193792131212)
     | > log_mle: 0.5747567415237427  (0.6339659955766466)
     | > loss_dur: 1.8642197847366333  (1.9132277965545654)
     | > amp_scaler: 4096.0  (4096.0)
     | > grad_norm: tensor(17.9378, device='cuda:0')  (tensor(19.0749, device='cuda:0'))
     | > current_lr: 5e-09 
     | > step_time: 0.318  (0.3015511565738254)
    


  --> EVAL PERFORMANCE
     | > avg_loader_time: 0.13478195667266846 (-0.012224912643432617)
     | > avg_loss: 2.4126272201538086 (-0.005839824676513672)
     | > avg_log_mle: 0.7780387699604034 (-0.0024688243865966797)
     | > avg_loss_dur: 1.6345884203910828 (-0.0033710598945617676)

 > BEST MODEL : C:/TTS-glowtts/output\glow-tts-rel_pos_transformer-February-22-2026_06+45PM-0000000\best_model_699.pth

 > EPOCH: 3/100
 --> C:/TTS-glowtts/output\glow-tts-rel_pos_transformer-February-22-2026_06+45PM-0000000

 > TRAINING (2026-02-22 18:55:53) 

   --> TIME: 2026-02-22 18:55:54 -- STEP: 1/233 -- GLOBAL_STEP: 700
     | > loss: 2.467939853668213  (2.467939853668213)
     | > log_mle: 0.6540114879608154  (0.6540114879608154)
     | > loss_dur: 1.813928484916687  (1.813928484916687)
     | > amp_scaler: 4096.0  (4096.0)
     | > grad_norm: tensor(21.0272, device='cuda:0')  (tensor(21.0272, device='cuda:0'))
     | > current_lr: 7.500000000000001e-09 
     | > step_time: 0.3246  (0.3245906

...


  --> EVAL PERFORMANCE
     | > avg_loader_time: 0.11207866668701172 (+0.005181670188903809)
     | > avg_loss: 0.842398464679718 (-0.024325013160705566)
     | > avg_log_mle: -0.3007326126098633 (-0.008811235427856445)
     | > avg_loss_dur: 1.1431310772895813 (-0.015513777732849121)

 > BEST MODEL : C:/TTS-glowtts/output\glow-tts-rel_pos_transformer-February-22-2026_06+45PM-0000000\best_model_22368.pth

 > EPOCH: 96/100
 --> C:/TTS-glowtts/output\glow-tts-rel_pos_transformer-February-22-2026_06+45PM-0000000

 > TRAINING (2026-02-22 22:32:58) 

   --> TIME: 2026-02-22 22:33:01 -- STEP: 7/233 -- GLOBAL_STEP: 22375
     | > loss: 1.001275897026062  (1.167477743966239)
     | > log_mle: -0.2583707571029663  (-0.2637821435928345)
     | > loss_dur: 1.2596466541290283  (1.4312598875590734)
     | > amp_scaler: 8192.0  (8192.0)
     | > grad_norm: tensor(2.4144, device='cuda:0')  (tensor(2.6365, device='cuda:0'))
     | > current_lr: 2.4000000000000003e-07 
     | > step_time: 0.3017  (0.3


  --> EVAL PERFORMANCE
     | > avg_loader_time: 0.10423541069030762 (-0.007843255996704102)
     | > avg_loss: 0.8139210939407349 (-0.028477370738983154)
     | > avg_log_mle: -0.3094678521156311 (-0.008735239505767822)
     | > avg_loss_dur: 1.123388946056366 (-0.019742131233215332)

 > BEST MODEL : C:/TTS-glowtts/output\glow-tts-rel_pos_transformer-February-22-2026_06+45PM-0000000\best_model_22601.pth

 > EPOCH: 97/100
 --> C:/TTS-glowtts/output\glow-tts-rel_pos_transformer-February-22-2026_06+45PM-0000000

 > TRAINING (2026-02-22 22:35:11) 

   --> TIME: 2026-02-22 22:35:20 -- STEP: 24/233 -- GLOBAL_STEP: 22625
     | > loss: 1.0305981636047363  (1.0781697432200115)
     | > log_mle: -0.30003488063812256  (-0.28118499616781867)
     | > loss_dur: 1.3306330442428589  (1.35935473938783)
     | > amp_scaler: 8192.0  (8192.0)
     | > grad_norm: tensor(2.3449, device='cuda:0')  (tensor(2.4942, device='cuda:0'))
     | > current_lr: 2.425e-07 
     | > step_time: 0.3051  (0.29899470011


  --> EVAL PERFORMANCE
     | > avg_loader_time: 0.10706889629364014 (+0.0028334856033325195)
     | > avg_loss: 0.7891554236412048 (-0.02476567029953003)
     | > avg_log_mle: -0.31817150115966797 (-0.008703649044036865)
     | > avg_loss_dur: 1.1073269248008728 (-0.016062021255493164)

 > BEST MODEL : C:/TTS-glowtts/output\glow-tts-rel_pos_transformer-February-22-2026_06+45PM-0000000\best_model_22834.pth

 > EPOCH: 98/100
 --> C:/TTS-glowtts/output\glow-tts-rel_pos_transformer-February-22-2026_06+45PM-0000000

 > TRAINING (2026-02-22 22:37:22) 

   --> TIME: 2026-02-22 22:37:29 -- STEP: 16/233 -- GLOBAL_STEP: 22850
     | > loss: 1.1831562519073486  (1.0851458087563515)
     | > log_mle: -0.3015786409378052  (-0.2871423587203025)
     | > loss_dur: 1.4847348928451538  (1.372288167476654)
     | > amp_scaler: 8192.0  (8192.0)
     | > grad_norm: tensor(2.5615, device='cuda:0')  (tensor(2.5555, device='cuda:0'))
     | > current_lr: 2.45e-07 
     | > step_time: 0.2888  (0.28882232308


  --> EVAL PERFORMANCE
     | > avg_loader_time: 0.10071837902069092 (-0.006350517272949219)
     | > avg_loss: 0.7704318761825562 (-0.01872354745864868)
     | > avg_log_mle: -0.3268086314201355 (-0.00863713026046753)
     | > avg_loss_dur: 1.0972405076026917 (-0.010086417198181152)

 > BEST MODEL : C:/TTS-glowtts/output\glow-tts-rel_pos_transformer-February-22-2026_06+45PM-0000000\best_model_23067.pth

 > EPOCH: 99/100
 --> C:/TTS-glowtts/output\glow-tts-rel_pos_transformer-February-22-2026_06+45PM-0000000

 > TRAINING (2026-02-22 22:39:34) 

   --> TIME: 2026-02-22 22:39:37 -- STEP: 8/233 -- GLOBAL_STEP: 23075
     | > loss: 1.2898505926132202  (1.0921127200126648)
     | > log_mle: -0.31246471405029297  (-0.2924404740333557)
     | > loss_dur: 1.6023153066635132  (1.3845531940460205)
     | > amp_scaler: 8192.0  (8192.0)
     | > grad_norm: tensor(2.4978, device='cuda:0')  (tensor(2.5907, device='cuda:0'))
     | > current_lr: 2.475e-07 
     | > step_time: 0.2677  (0.277704805135


  --> EVAL PERFORMANCE
     | > avg_loader_time: 0.11614131927490234 (+0.015422940254211426)
     | > avg_loss: 0.7589383721351624 (-0.011493504047393799)
     | > avg_log_mle: -0.33534765243530273 (-0.008539021015167236)
     | > avg_loss_dur: 1.094286024570465 (-0.0029544830322265625)

 > BEST MODEL : C:/TTS-glowtts/output\glow-tts-rel_pos_transformer-February-22-2026_06+45PM-0000000\best_model_23300.pth
